In [1]:
import torch
import torchvision
from torchvision import datasets
from torchvision import transforms
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim

In [2]:
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

device = torch.device('cuda' if torch.cuda.is_available()else'cpu')

PyTorch: 2.5.1
Torchvision: 0.20.1
CUDA available: True
GPU: NVIDIA GeForce 940MX


In [3]:
train_dataset = datasets.ImageFolder(root=r"F:\Data Science\Projects\Real vs AI\train")
test_dataset = datasets.ImageFolder(root=r"F:\Data Science\Projects\Real vs AI\test")

In [4]:
print(f'Train Classes: {train_dataset.classes}')
print(f'Test Classes: {test_dataset.classes}')

Train Classes: ['FAKE', 'REAL']
Test Classes: ['FAKE', 'REAL']


In [5]:
print(f'Train Classes: {train_dataset.class_to_idx}')
print(f'Test Classes: {train_dataset.class_to_idx}')

Train Classes: {'FAKE': 0, 'REAL': 1}
Test Classes: {'FAKE': 0, 'REAL': 1}


In [6]:
print("Training images:", len(train_dataset))
print("Testing images:", len(test_dataset))

Training images: 100000
Testing images: 20000


In [7]:
image, label = train_dataset[0]

print("Image:", image)
print("Label:", label)

Image: <PIL.Image.Image image mode=RGB size=32x32 at 0x24D1B38EF10>
Label: 0


In [8]:
print(type(image))
print(image.size)
print(train_dataset.classes[label])

<class 'PIL.Image.Image'>
(32, 32)
FAKE


In [9]:
transform = transforms.Compose([
    # transforms.Resize((224,224)),
    transforms.ToTensor()
])

In [10]:
train_dataset = datasets.ImageFolder(root=r"F:\Data Science\Projects\Real vs AI\train", transform = transform)
test_dataset = datasets.ImageFolder(root=r"F:\Data Science\Projects\Real vs AI\test", transform = transform)

In [11]:
image, label = train_dataset[0]

print("Image:", image)
print("Label:", label)
print(type(image))
print(image.shape)

Image: tensor([[[0.4471, 0.4667, 0.4588,  ..., 0.4863, 0.4157, 0.2118],
         [0.4980, 0.4980, 0.4863,  ..., 0.4039, 0.3176, 0.1804],
         [0.5216, 0.5020, 0.4941,  ..., 0.2824, 0.1961, 0.1569],
         ...,
         [0.1373, 0.1412, 0.1529,  ..., 0.3490, 0.3451, 0.3490],
         [0.1333, 0.1490, 0.1569,  ..., 0.3647, 0.3451, 0.3412],
         [0.1373, 0.1451, 0.1569,  ..., 0.3922, 0.3882, 0.3843]],

        [[0.4392, 0.4588, 0.4588,  ..., 0.4706, 0.4000, 0.1961],
         [0.4902, 0.4902, 0.4863,  ..., 0.3882, 0.3020, 0.1647],
         [0.5137, 0.4941, 0.4863,  ..., 0.2667, 0.1804, 0.1412],
         ...,
         [0.1098, 0.1137, 0.1176,  ..., 0.3216, 0.3176, 0.3216],
         [0.1176, 0.1216, 0.1294,  ..., 0.3294, 0.3137, 0.3098],
         [0.1176, 0.1255, 0.1412,  ..., 0.3569, 0.3569, 0.3529]],

        [[0.4431, 0.4627, 0.4510,  ..., 0.4667, 0.3961, 0.1922],
         [0.4941, 0.4941, 0.4863,  ..., 0.3843, 0.2980, 0.1608],
         [0.5176, 0.4980, 0.4902,  ..., 0.2549, 0.1

In [12]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle = True, pin_memory= True, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle = False, pin_memory= True, num_workers=4)

In [13]:
images, labels = next(iter(train_loader))

print("Images shape:", images.shape)
print("Labels shape:", labels.shape)
print("Labels:", labels)

Images shape: torch.Size([64, 3, 32, 32])
Labels shape: torch.Size([64])
Labels: tensor([1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0,
        0, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 0, 1, 1, 1, 0, 0, 0,
        0, 1, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1])


In [14]:
class MYCNN(nn.Module):
    def __init__(self, input_features):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(input_features, 32, kernel_size=3, padding='same'),
            nn.ReLU(),
            nn.BatchNorm2d(32),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(32, 64, kernel_size=3, padding='same'),
            nn.ReLU(),
            nn.BatchNorm2d(64),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*8*8, 128),
            nn.ReLU(),
            nn.Linear(128,64),
            nn.ReLU(),
            nn.Linear(64,2),
        )
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)

        return x

In [15]:
learning_rate = 0.1
epochs = 20

In [26]:
model = MYCNN(3)
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = learning_rate)

In [38]:
print(model)

MYCNN(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=same)
    (1): ReLU()
    (2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=same)
    (5): ReLU()
    (6): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=4096, out_features=128, bias=True)
    (2): ReLU()
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): ReLU()
    (5): Linear(in_features=64, out_features=2, bias=True)
  )
)


In [17]:
# Training Loop

'''for epoch in range(epochs):
    total_epoch_loss = 0
    
    for batch_features, batch_labels in train_loader:

        # MOve to GPU
        batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

        # forward Pass
        output = model(batch_features)
        
        # calculate Loss
        loss = criterion(output, batch_labels)
        
        # Backpropogation
        optimizer.zero_grad()
        loss.backward()
        
        # Update Grads
        optimizer.step()
        
        total_epoch_loss = total_epoch_loss + loss.item()

    avg_loss = total_epoch_loss/len(train_loader)
    print(f"Epoch: {epoch+1}, Loss: {avg_loss}")'''

'for epoch in range(epochs):\n    total_epoch_loss = 0\n\n    for batch_features, batch_labels in train_loader:\n\n        # MOve to GPU\n        batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)\n\n        # forward Pass\n        output = model(batch_features)\n\n        # calculate Loss\n        loss = criterion(output, batch_labels)\n\n        # Backpropogation\n        optimizer.zero_grad()\n        loss.backward()\n\n        # Update Grads\n        optimizer.step()\n\n        total_epoch_loss = total_epoch_loss + loss.item()\n\n    avg_loss = total_epoch_loss/len(train_loader)\n    print(f"Epoch: {epoch+1}, Loss: {avg_loss}")'